In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor, DMatrix

In [2]:
RANDOM_STATE = 12345

# Load data

In [3]:
housing_data = fetch_california_housing(as_frame=True)
housing_df = housing_data.frame
feature_cols = housing_data.feature_names
target_cols = housing_data.target_names

housing_df.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [4]:
X = housing_df[feature_cols].values
y = housing_df[target_cols].squeeze()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
X_valid, X_test, y_valid, y_test = train_test_split(X_test, y_test, test_size=0.5, random_state=RANDOM_STATE)

print('Train', X_train.shape, y_train.shape)
print('Valid', X_valid.shape, y_valid.shape)
print('Test', X_test.shape, y_test.shape)

Train (16512, 8) (16512,)
Valid (2064, 8) (2064,)
Test (2064, 8) (2064,)


# XGBRegressor

In [5]:
params = {
    'booster': 'gbtree',
    'objective': 'reg:squarederror',  # regression with squared log loss
    'learning_rate': 0.3,
    'max_depth': 3,
    'n_estimators': 3,
    'reg_lambda': 1,                  # default for gbtree booster
    'gamma': 0,
    'tree_method': 'exact',           # Exact greedy algorithm. Enumerates all split candidates.
    'min_child_weight': 1,
    'random_state': RANDOM_STATE,
}

# Instantiate XGBRegressor with parameters
regressor = XGBRegressor(**params)

# Train the model using fit() method
regressor.fit(X_train, y_train)
booster = regressor.get_booster()
tree_dump = booster.get_dump(with_stats=True)

In [6]:
print(tree_dump[0])

0:[f0<5.07534981] yes=1,no=2,missing=1,gain=6792.41113,cover=16512
	1:[f0<3.12879992] yes=3,no=4,missing=3,gain=1729.58997,cover=13128
		3:[f2<4.31125164] yes=7,no=8,missing=7,gain=348.441162,cover=6577
			7:leaf=-0.123906545,cover=2714
			8:leaf=-0.264238626,cover=3863
		4:[f5<2.34443903] yes=9,no=10,missing=9,gain=1025.93201,cover=6551
			9:leaf=0.237810686,cover=1412
			10:leaf=-0.0508238673,cover=5139
	2:[f0<6.59465027] yes=5,no=6,missing=5,gain=1248.26953,cover=3384
		5:[f5<2.58064175] yes=11,no=12,missing=11,gain=378.475464,cover=2167
			11:leaf=0.439879924,cover=621
			12:leaf=0.162545979,cover=1546
		6:[f0<7.81515026] yes=13,no=14,missing=13,gain=251.017578,cover=1217
			13:leaf=0.481786162,cover=598
			14:leaf=0.756417394,cover=619



In [7]:
print(tree_dump[1])

0:[f0<4.39599991] yes=1,no=2,missing=1,gain=3573.4812,cover=16512
	1:[f5<2.26237345] yes=3,no=4,missing=3,gain=691.296875,cover=11474
		3:[f6<37.9349976] yes=7,no=8,missing=7,gain=355.934937,cover=2194
			7:leaf=0.120267555,cover=1745
			8:leaf=-0.178971276,cover=449
		4:[f0<2.51959991] yes=9,no=10,missing=9,gain=416.575806,cover=9280
			9:leaf=-0.215978563,cover=3194
			10:leaf=-0.0821812898,cover=6086
	2:[f0<6.32264996] yes=5,no=6,missing=5,gain=976.786133,cover=5038
		5:[f5<2.73610783] yes=11,no=12,missing=11,gain=389.850403,cover=3590
			11:leaf=0.243375391,cover=1499
			12:leaf=0.0429230221,cover=2091
		6:[f1<26.5] yes=13,no=14,missing=13,gain=175.402344,cover=1448
			13:leaf=0.32525745,cover=811
			14:leaf=0.536620557,cover=637



In [8]:
print(tree_dump[2])

0:[f0<3.93484998] yes=1,no=2,missing=1,gain=1865,cover=16512
	1:[f5<2.04001665] yes=3,no=4,missing=3,gain=311.742798,cover=9802
		3:[f0<2.44659996] yes=7,no=8,missing=7,gain=142.118149,cover=1108
			7:leaf=-0.0761050135,cover=401
			8:leaf=0.147279337,cover=707
		4:[f6<34.4749985] yes=9,no=10,missing=9,gain=213.198669,cover=8694
			9:leaf=-0.0546087287,cover=4267
			10:leaf=-0.148597509,cover=4427
	2:[f0<5.7873497] yes=5,no=6,missing=5,gain=640.238281,cover=6710
		5:[f5<2.44516444] yes=11,no=12,missing=11,gain=356.605774,cover=4636
			11:leaf=0.210405812,cover=1084
			12:leaf=0.0138922101,cover=3552
		6:[f0<7.95809984] yes=13,no=14,missing=13,gain=194.892334,cover=2074
			13:leaf=0.20367606,cover=1506
			14:leaf=0.410146028,cover=568



### Notes
XGBoost trees are built with a different split criterion (gradient/hessian-based gain, second-order Newton leaf values) than sklearn's DecisionTreeClassifier/Regressor (which use Gini/MSE).

# Algorithm

**Initialize with a Baseline Prediction ($F_0$)**  
Start with a constant prediction that minimizes the overall loss function.
For **squared-error loss**, this constant is the **mean of the target values**:
$$F_0 = \frac{1}{n}\sum_{i=1}^{n} y_i$$

In [9]:
F0 = np.full_like(y_train, fill_value=np.mean(y_train))
F = F0

**Calculate Pseudo-Residuals:**  
Compute the negative gradient of the loss function with respect to the current predictions.
$$ PseudoResiduals_i = - (predict_i - observe_i) = observe_i - predict_i $$

In [10]:
residuals = y_train - F

**Fit a New Tree:**  
Train a decision tree to predict these pseudo-residuals.

In [11]:
def similarity_score(residual, reg_lambda=1):
    sum_gradient = np.sum(residual) 
    sum_hessian = len(residual)
    return sum_gradient**2/(sum_hessian + reg_lambda)

def leaf_value(residual, reg_lambda=1, learning_rate=0.3):
    return learning_rate*np.sum(residual)/(len(residual) + reg_lambda)
    
def find_best_split(X, residuals, feature_cols, gamma=0):
    
    similarity_score_parent = similarity_score(residuals)

    max_gain = -1
    best_feature_idx = None
    best_feature = None
    best_thres = None
    for idx in range(len(feature_cols)):
        # For each feature, sort unique values
        Xf = X[:, idx]
        sort_unq_Xf = np.sort(np.unique(Xf))

        # find possible threshold between consecutive sorted values
        possible_thresholds = (sort_unq_Xf[:-1] + sort_unq_Xf[1:])/2 

        for thres in possible_thresholds:
            # For each possible threshold, splitting the node into a "left" group and a "right" group
            left_grp = residuals[Xf <= thres]
            right_grp = residuals[Xf > thres]

            similarity_score_left_grp = similarity_score(left_grp)
            similarity_score_right_grp = similarity_score(right_grp)
    
            gain = similarity_score_left_grp + similarity_score_right_grp - similarity_score_parent
            if (gain > max_gain) and (gain > 0):
                max_gain = gain
                best_feature_idx = idx
                best_feature = feature_cols[idx]
                best_thres = thres
                best_left_grp = [X[Xf <= thres,:], left_grp]
                best_right_grp = [X[Xf > thres,:], right_grp]

    return best_feature, best_feature_idx, best_thres, best_left_grp, best_right_grp, max_gain

In [12]:
f, fidx, thres, left_grp, right_grp, gain = find_best_split(X_train, residuals, feature_cols)
print(f'0:[f{fidx}<={thres:.8f}], gain={gain}')

f, fidx, thres, left_grp10, right_grp10, gain = find_best_split(left_grp[0], left_grp[1], feature_cols)
print(f'	1:[f{fidx}<={thres:.8f}], gain={gain}')

f, fidx, thres, left_grp100, right_grp100, gain = find_best_split(left_grp10[0], left_grp10[1], feature_cols)
print(f'		3:[f{fidx}<={thres:.8f}], gain={gain}')
print(f'			7:leaf={leaf_value(left_grp100[1])}')
print(f'			8:leaf={leaf_value(right_grp100[1])}')

f, fidx, thres, left_grp101, right_grp101, gain = find_best_split(right_grp10[0], right_grp10[1], feature_cols)
print(f'		4:[f{fidx}<={thres:.8f}], gain={gain}')
print(f'			9:leaf={leaf_value(left_grp101[1])}')
print(f'			10:leaf={leaf_value(right_grp101[1])}')

f, fidx, thres, left_grp10, right_grp10, gain = find_best_split(right_grp[0], right_grp[1], feature_cols)
print(f'	2:f{fidx}<={thres:.8f}], gain={gain}')

f, fidx, thres, left_grp100, right_grp100, gain = find_best_split(left_grp10[0], left_grp10[1], feature_cols)
print(f'		5:[f{fidx}<={thres:.8f}], gain={gain}')
print(f'			11:leaf={leaf_value(left_grp100[1])}')
print(f'			12:leaf={leaf_value(right_grp100[1])}')

f, fidx, thres, left_grp101, right_grp101, gain = find_best_split(right_grp10[0], right_grp10[1], feature_cols)
print(f'		6:[f{fidx}<={thres:.8f}], gain={gain}')
print(f'			13:leaf={leaf_value(left_grp101[1])}')
print(f'			14:leaf={leaf_value(right_grp101[1])}')

0:[f0<=5.07535000], gain=6792.411239257684
	1:[f0<=3.12880000], gain=1729.5902256938716
		3:[f2<=4.31125163], gain=348.4409664072241
			7:leaf=-0.12390654783047449
			8:leaf=-0.26423862353118294
		4:[f5<=2.34443898], gain=1025.9320491760634
			9:leaf=0.2378106670009793
			10:leaf=-0.0508238747455688
	2:f0<=6.59465000], gain=1248.2691535150188
		5:[f5<=2.58064165], gain=378.47560105045295
			11:leaf=0.4398799224656257
			12:leaf=0.16254596278289943
		6:[f0<=7.81515000], gain=251.01758107861588
			13:leaf=0.48178614493996774
			14:leaf=0.7564173312681687


**Update Predictions:**  
Add the new tree's scaled leaf outputs to the current predictions:

 $$F_m(x) = F_{m-1}(x) + \eta f_m(x)$$

 where $\eta$ is the **learning rate** and $f_m(x)$ is the newly trained tree.

In [13]:
F1 = booster.predict(DMatrix(X_train), iteration_range=(0, 1)) #[start, end)

**Repeat Until the Stopping Condition Is Met at n_estimators=3**  

In [14]:
residuals = y_train - F1

f, fidx, thres, left_grp, right_grp, gain = find_best_split(X_train, residuals, feature_cols)
print(f'0:[f{fidx}<={thres:.8f}], gain={gain}')

f, fidx, thres, left_grp10, right_grp10, gain = find_best_split(left_grp[0], left_grp[1], feature_cols)
print(f'	1:[f{fidx}<={thres:.8f}], gain={gain}')

f, fidx, thres, left_grp100, right_grp100, gain = find_best_split(left_grp10[0], left_grp10[1], feature_cols)
print(f'		3:[f{fidx}<={thres:.8f}], gain={gain}')
print(f'			7:leaf={leaf_value(left_grp100[1])}')
print(f'			8:leaf={leaf_value(right_grp100[1])}')

f, fidx, thres, left_grp101, right_grp101, gain = find_best_split(right_grp10[0], right_grp10[1], feature_cols)
print(f'		4:[f{fidx}<={thres:.8f}], gain={gain}')
print(f'			9:leaf={leaf_value(left_grp101[1])}')
print(f'			10:leaf={leaf_value(right_grp101[1])}')

f, fidx, thres, left_grp10, right_grp10, gain = find_best_split(right_grp[0], right_grp[1], feature_cols)
print(f'	2:f{fidx}<={thres:.8f}], gain={gain}')

f, fidx, thres, left_grp100, right_grp100, gain = find_best_split(left_grp10[0], left_grp10[1], feature_cols)
print(f'		5:[f{fidx}<={thres:.8f}], gain={gain}')
print(f'			11:leaf={leaf_value(left_grp100[1])}')
print(f'			12:leaf={leaf_value(right_grp100[1])}')

f, fidx, thres, left_grp101, right_grp101, gain = find_best_split(right_grp10[0], right_grp10[1], feature_cols)
print(f'		6:[f{fidx}<={thres:.8f}], gain={gain}')
print(f'			13:leaf={leaf_value(left_grp101[1])}')
print(f'			14:leaf={leaf_value(right_grp101[1])}')

0:[f0<=4.39600000], gain=3573.481210766634
	1:[f5<=2.26237362], gain=691.296792465133
		3:[f6<=37.93500000], gain=355.93495873022323
			7:leaf=0.12026755186757353
			8:leaf=-0.17897126771481833
		4:[f0<=2.51960000], gain=416.5758827877605
			9:leaf=-0.21597855718576134
			10:leaf=-0.08218128783652512
	2:f0<=6.32265000], gain=976.7862824372005
		5:[f5<=2.73610781], gain=389.8503660241744
			11:leaf=0.24337537952743532
			12:leaf=0.042923021888320805
		6:[f1<=26.50000000], gain=175.4021044882743
			13:leaf=0.32525744784787364
			14:leaf=0.5366205071906923


In [15]:
F2 = booster.predict(DMatrix(X_train), iteration_range=(0, 2)) #[start, end)

In [16]:
residuals = y_train - F2

f, fidx, thres, left_grp, right_grp, gain = find_best_split(X_train, residuals, feature_cols)
print(f'0:[f{fidx}<={thres:.8f}], gain={gain}')

f, fidx, thres, left_grp10, right_grp10, gain = find_best_split(left_grp[0], left_grp[1], feature_cols)
print(f'	1:[f{fidx}<={thres:.8f}], gain={gain}')

f, fidx, thres, left_grp100, right_grp100, gain = find_best_split(left_grp10[0], left_grp10[1], feature_cols)
print(f'		3:[f{fidx}<={thres:.8f}], gain={gain}')
print(f'			7:leaf={leaf_value(left_grp100[1])}')
print(f'			8:leaf={leaf_value(right_grp100[1])}')

f, fidx, thres, left_grp101, right_grp101, gain = find_best_split(right_grp10[0], right_grp10[1], feature_cols)
print(f'		4:[f{fidx}<={thres:.8f}], gain={gain}')
print(f'			9:leaf={leaf_value(left_grp101[1])}')
print(f'			10:leaf={leaf_value(right_grp101[1])}')

f, fidx, thres, left_grp10, right_grp10, gain = find_best_split(right_grp[0], right_grp[1], feature_cols)
print(f'	2:f{fidx}<={thres:.8f}], gain={gain}')

f, fidx, thres, left_grp100, right_grp100, gain = find_best_split(left_grp10[0], left_grp10[1], feature_cols)
print(f'		5:[f{fidx}<={thres:.8f}], gain={gain}')
print(f'			11:leaf={leaf_value(left_grp100[1])}')
print(f'			12:leaf={leaf_value(right_grp100[1])}')

f, fidx, thres, left_grp101, right_grp101, gain = find_best_split(right_grp10[0], right_grp10[1], feature_cols)
print(f'		6:[f{fidx}<={thres:.8f}], gain={gain}')
print(f'			13:leaf={leaf_value(left_grp101[1])}')
print(f'			14:leaf={leaf_value(right_grp101[1])}')

0:[f0<=3.93485000], gain=1864.9999512965417
	1:[f5<=2.04001668], gain=311.7428321758979
		3:[f0<=2.44660000], gain=142.1181520172046
			7:leaf=-0.07610500670754732
			8:leaf=0.1472793313715498
		4:[f6<=34.47500000], gain=213.19865981909425
			9:leaf=-0.054608723184936835
			10:leaf=-0.14859751100717197
	2:f0<=5.78735000], gain=640.2384421949982
		5:[f5<=2.44516441], gain=356.6057652759347
			11:leaf=0.21040579842368623
			12:leaf=0.013892210584702235
		6:[f0<=7.95810000], gain=194.89233504823233
			13:leaf=0.20367603530650397
			14:leaf=0.41014602366685105
